[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/mathematical_reasoning/06_asymptotics_and_algorithmic_reasoning/first_principles.ipynb)

# Topic 06: Asymptotics and Algorithmic Reasoning

## 1. First-Principles Intuition & Motivation

Suppose you time two sorting routines. On your laptop, with your compiler, on your test data, routine A takes $0.8$ seconds and routine B takes $1.9$ seconds. What have you learned? Almost nothing transferable: change the machine, the language, the cache size, or the input distribution and the ordering can reverse. What *is* transferable is how each routine's cost responds to the size of the input. If A doubles when the input doubles and B quadruples, then no constant factor, however lovingly optimized, will save B at scale. Asymptotic analysis is the deliberate decision to study that response and to throw away everything else.

Throwing information away is not a defect; it is the point. The discarded constants are exactly the machine-, compiler-, and implementation-dependent quantities, while the retained growth rate is a property of the *algorithm*. This is why $\Theta(n\log n)$ is a statement that has held for merge sort since 1945 and will hold on hardware not yet designed, while "merge sort takes $4.7n\log_2 n + 312n$ nanoseconds" is a statement about one afternoon.

The formal apparatus is a set of relations between functions, each defined by an explicit quantifier statement:

- $f = O(g)$: $f$ grows *no faster* than $g$, up to a constant — a guarantee, an upper bound, a promise.
- $f = \Omega(g)$: $f$ grows *at least* as fast as $g$ — an impossibility result, a lower bound.
- $f = \Theta(g)$: both at once — the honest description of an algorithm's cost.
- $f = o(g)$ and $f = \omega(g)$: strict versions, where the ratio tends to $0$ or $\infty$ — used for "asymptotically negligible" and "asymptotically dominant".

The single most common misuse is treating $O$ as though it were $\Theta$. The statement "merge sort is $O(n^{5})$" is *true*. It is also useless, and it is not wrong — which is precisely why one must say $\Theta$ when one means tight. Note also that the case (worst, best, average) is an independent axis: quicksort is $\Theta(n^{2})$ in the worst case and $\Theta(n\log n)$ in expectation, and neither statement contradicts the other.

Getting a growth rate in the first place is the second half of the subject. Loops are summed; recursion produces recurrences, and recurrences are solved by three interlocking methods:

1. **Substitution** — guess the answer and verify it by induction (Topic 04's machinery applied verbatim). It is the only method that produces a certified proof, and it has one recurring trap: a hypothesis that is *too weak* fails to carry the induction, and the fix is to strengthen it by subtracting a lower-order term.
2. **Recursion tree** — draw the tree of recursive calls, sum the work at each level, and sum across levels. This method both *finds* the answer and, when the levels form a geometric series, proves it.
3. **Master theorem** — a packaged answer for $T(n) = aT(n/b) + f(n)$, obtained by running the recursion-tree argument once and for all. Its three cases correspond to the three ways a geometric series can behave: dominated by the leaves, evenly spread, or dominated by the root.

Finally, **amortized analysis** answers a question that per-operation worst-case bounds cannot: a dynamic array append is occasionally $\Theta(n)$, yet $m$ appends cost $\Theta(m)$ in total. Amortization is not averaging over randomness — it is a deterministic accounting argument, and the potential method makes it rigorous by defining a bank account whose balance absorbs the expensive operations.

Why this matters for machine learning. Every architectural decision has an asymptotic signature: attention over $n$ tokens costs $\Theta(n^{2}d)$ time and $\Theta(n^{2})$ memory for the score matrix, feed-forward layers cost $\Theta(nd^{2})$, so the crossover at $n \approx d$ determines whether context length or model width dominates. Backpropagation costs a small constant times the forward pass, not a factor of the number of parameters — a fact (Baur–Strassen) without which deep learning would be infeasible. Training a transformer costs about $6N$ FLOPs per token, which is the entire basis of compute-budget planning. And the empirical fact that modern accelerators are memory-bandwidth-bound rather than FLOP-bound means the right cost model is often *bytes moved*, not operations performed — an asymptotic analysis of the wrong resource is worse than none.

## 2. Rigorous Mathematical Definitions & Theorem Statements

Throughout, $f, g : \mathbb{N} \to \mathbb{R}_{\geq 0}$ are eventually nonnegative functions.

**Definition 2.1 (Big-O).** $f(n) = O\bigl(g(n)\bigr)$ iff there exist constants $c \gt 0$ and $n_0 \in \mathbb{N}$ such that

$$
0 \le f(n) \le c\,g(n) \quad \text{for all } n \ge n_0
$$

**Definition 2.2 (Big-Omega).** $f(n) = \Omega\bigl(g(n)\bigr)$ iff there exist $c \gt 0$ and $n_0$ with $0 \le c\,g(n) \le f(n)$ for all $n \ge n_0$.

**Definition 2.3 (Theta).** $f(n) = \Theta\bigl(g(n)\bigr)$ iff there exist $c_1, c_2 \gt 0$ and $n_0$ with

$$
0 \lt c_1\,g(n) \le f(n) \le c_2\,g(n) \quad \text{for all } n \ge n_0
$$

Equivalently, $f = O(g)$ and $f = \Omega(g)$.

**Definition 2.4 (little-o, little-omega).** $f(n) = o(g(n))$ iff for *every* $c \gt 0$ there is an $n_0$ with $f(n) \lt c\,g(n)$ for $n \ge n_0$; when $g$ is eventually positive this is equivalent to $\lim_{n\to\infty} f(n)/g(n) = 0$. Dually $f = \omega(g)$ iff $g = o(f)$.

**Remark 2.5 (the "equals" sign is an abuse).** $f = O(g)$ means $f \in O(g)$, where $O(g)$ is a *set* of functions. The notation is one-directional: one writes $2n^{2} + n = O(n^{2})$ but never $O(n^{2}) = 2n^{2} + n$. Inside an expression, $O(g)$ denotes "some anonymous function in $O(g)$", so $T(n) = 2T(n/2) + O(n)$ is a legitimate and standard idiom.

**Proposition 2.6 (basic algebra of $O$).** For eventually nonnegative $f, f_1, f_2, g$:

| Rule | Statement |
|---|---|
| Reflexivity | $f = O(f)$, $f = \Theta(f)$ |
| Transitivity | $f = O(g)$ and $g = O(h)$ imply $f = O(h)$ |
| Sum | $O(f_1) + O(f_2) = O(\max(f_1, f_2))$ |
| Product | $f_1 = O(g_1)$ and $f_2 = O(g_2)$ imply $f_1 f_2 = O(g_1 g_2)$ |
| Constant absorption | $O(c\,f) = O(f)$ for constant $c \gt 0$ |
| Symmetry of $\Theta$ | $f = \Theta(g)$ iff $g = \Theta(f)$ |
| Duality | $f = O(g)$ iff $g = \Omega(f)$ |

**Theorem 2.7 (growth hierarchy).** For constants $\epsilon \gt 0$, $k \gt 0$, $a \gt 1$:

$$
1 \prec \log\log n \prec \log n \prec (\log n)^{k} \prec n^{\epsilon} \prec n^{k} \prec a^{n} \prec n! \prec n^{n}
$$

where $u \prec v$ means $u = o(v)$. In particular polylogarithms are $o$ of every positive power, and every polynomial is $o$ of every exponential.

**Theorem 2.8 (Master theorem).** Let $a \ge 1$, $b \gt 1$ be constants, $f(n)$ nonnegative, and

$$
T(n) = a\,T(n/b) + f(n)
$$

Write $\alpha = \log_b a$, so $n^{\alpha}$ is the total leaf work. Then:

1. **Leaf-dominated.** If $f(n) = O(n^{\alpha - \epsilon})$ for some $\epsilon \gt 0$, then $T(n) = \Theta(n^{\alpha})$.
2. **Balanced.** If $f(n) = \Theta(n^{\alpha}\log^{k} n)$ for some $k \ge 0$, then $T(n) = \Theta(n^{\alpha}\log^{k+1} n)$.
3. **Root-dominated.** If $f(n) = \Omega(n^{\alpha + \epsilon})$ for some $\epsilon \gt 0$ *and* the regularity condition $a f(n/b) \le c f(n)$ holds for some $c \lt 1$ and large $n$, then $T(n) = \Theta(f(n))$.

**Definition 2.9 (amortized cost; potential method).** Let $\Phi$ map a data-structure state to a real number with $\Phi(D_0) = 0$ and $\Phi(D_i) \ge 0$ for all $i$. The *amortized cost* of the $i$-th operation is

$$
\hat{c}_i = c_i + \Phi(D_i) - \Phi(D_{i-1})
$$

Then $\sum_{i=1}^{m} c_i = \sum_{i=1}^{m}\hat{c}_i - \Phi(D_m) + \Phi(D_0) \le \sum_{i=1}^{m}\hat{c}_i$, so any per-operation bound on $\hat{c}_i$ bounds the total actual cost.

**Theorem 2.10 (comparison-sorting lower bound).** Any deterministic algorithm that sorts $n$ elements using only pairwise comparisons performs $\Omega(n\log n)$ comparisons in the worst case.

**Statements proved in Section 3.**

| # | Statement | Instrument |
|---|---|---|
| 3.1 | $3n^{2} + 10n + 7 = \Theta(n^{2})$, and $n^{2} \neq O(n)$ | Explicit constants; contradiction |
| 3.2 | $\log n = o(n^{\epsilon})$, $n^{k} = o(a^{n})$, $\log(n!) = \Theta(n\log n)$ | Limits, ratio test, integral bounds |
| 3.3 | Master theorem | Recursion tree plus a geometric series |
| 3.4 | $T(n) = 2T(n/2) + n$ is $\Theta(n\log n)$ | Substitution, with the strengthened-hypothesis trap |
| 3.5 | Dynamic-array append is $O(1)$ amortized | Aggregate, accounting, and potential methods |
| 3.6 | Comparison sorting is $\Omega(n\log n)$ | Decision tree plus a counting argument (Topic 05) |

## 3. Step-by-Step Mathematical Proofs & Derivations

Every proof below either *exhibits witness constants* or *derives a contradiction from their existence* — those are the only two moves available when working directly from Definitions 2.1–2.4. When a proof does something else, it is silently invoking a previously proved lemma.

### Proof 3.1 — Tight Bounds Directly from the Definition

**Claim A.** $3n^{2} + 10n + 7 = \Theta(n^{2})$.

**Proof.** We exhibit $c_1, c_2, n_0$ satisfying Definition 2.3.

*Upper bound.* For $n \ge 1$ we have $10n \le 10n^{2}$ and $7 \le 7n^{2}$, so

$$
3n^{2} + 10n + 7 \le 3n^{2} + 10n^{2} + 7n^{2} = 20n^{2}
$$

Take $c_2 = 20$, valid for all $n \ge 1$.

*Lower bound.* All terms are nonnegative for $n \ge 0$, so $3n^{2} + 10n + 7 \ge 3n^{2}$. Take $c_1 = 3$, valid for all $n \ge 0$.

With $c_1 = 3$, $c_2 = 20$, $n_0 = 1$ the definition is satisfied. $\blacksquare$

**Claim B.** $n^{2} \neq O(n)$.

**Proof (contradiction).** Suppose there were $c \gt 0$ and $n_0$ with $n^{2} \le cn$ for all $n \ge n_0$. Dividing by $n \gt 0$ gives $n \le c$ for all $n \ge n_0$ — but $n$ is unbounded, so taking $n = \max(n_0, \lceil c \rceil + 1)$ yields $n \gt c$, a contradiction. Hence no such constants exist. $\blacksquare$

**Strategic reading.** A positive $O$-claim is proved by *producing* constants; a negative one by *refuting* all possible constants, which is a universally quantified statement and therefore a natural target for contradiction. The constants need not be tight or elegant — $c_2 = 20$ is wasteful but perfectly valid, and looking for the smallest constant is a different (and much harder) problem than proving the asymptotic bound.

### Proof 3.2 — The Growth Hierarchy

**Claim A.** For every $\epsilon \gt 0$: $\log n = o(n^{\epsilon})$.

**Proof.** Consider the continuous analogue and apply L'Hôpital's rule to $\ln x / x^{\epsilon}$ (both factors tend to $\infty$):

$$
\lim_{x\to\infty}\frac{\ln x}{x^{\epsilon}} = \lim_{x\to\infty}\frac{1/x}{\epsilon x^{\epsilon-1}} = \lim_{x\to\infty}\frac{1}{\epsilon x^{\epsilon}} = 0
$$

Since $\log_b n = \ln n / \ln b$ differs from $\ln n$ by a constant factor, the limit is $0$ for any base, so $\log n = o(n^{\epsilon})$. Applying the result with $\epsilon/k$ and raising to the $k$-th power gives $(\log n)^{k} = o(n^{\epsilon})$ as well. $\blacksquare$

**Corollary.** All logarithm bases are equivalent under $\Theta$: $\log_a n = \frac{\log_b n}{\log_b a} = \Theta(\log_b n)$. This is why one writes $O(\log n)$ with no base — but note that the base does *not* disappear from an exponent: $2^{n}$ and $3^{n}$ are not $\Theta$-equivalent, since $3^{n}/2^{n} = (1.5)^{n} \to \infty$.

**Claim B.** For every constant $k \gt 0$ and $a \gt 1$: $n^{k} = o(a^{n})$.

**Proof.** Let $r_n = n^{k}/a^{n}$ and consider the ratio of consecutive terms:

$$
\frac{r_{n+1}}{r_n} = \frac{(n+1)^{k}}{n^{k}}\cdot\frac{1}{a} = \Bigl(1 + \frac{1}{n}\Bigr)^{k}\frac{1}{a} \longrightarrow \frac{1}{a} \lt 1
$$

So for large $n$ the ratio is bounded by some $\rho \lt 1$, whence $r_n$ decays at least geometrically and tends to $0$. Therefore $n^{k} = o(a^{n})$. $\blacksquare$

**Claim C.** $\log(n!) = \Theta(n\log n)$.

**Proof.** *Upper bound:* $n! = \prod_{i=1}^{n} i \le n^{n}$, so $\log(n!) \le n\log n$.

*Lower bound:* keep only the top half of the factors, each at least $n/2$:

$$
n! \ge \prod_{i = \lceil n/2 \rceil}^{n} i \ge \Bigl(\frac{n}{2}\Bigr)^{n/2} \implies \log(n!) \ge \frac{n}{2}\bigl(\log n - 1\bigr)
$$

which is $\ge \frac{n\log n}{4}$ for $n \ge 4$. Both bounds are constant multiples of $n\log n$. $\blacksquare$ (Stirling's formula $n! \sim \sqrt{2\pi n}\,(n/e)^{n}$ gives the sharper $\log(n!) = n\log n - n\log e + \Theta(\log n)$.)

**Strategic reading.** Three different instruments for three different comparisons: L'Hôpital for log-versus-power, the ratio test for power-versus-exponential, and crude product bounds for factorials. Claim C is the key input to the sorting lower bound of Proof 3.6.

### Proof 3.3 — The Master Theorem by Recursion Tree

**Setup.** Assume $n = b^{L}$ so that $L = \log_b n$ levels arise with no floor/ceiling complications (the general case follows by a standard but tedious argument). Unrolling $T(n) = aT(n/b) + f(n)$ produces a tree in which level $j$ contains $a^{j}$ nodes, each of input size $n/b^{j}$, contributing work $f(n/b^{j})$. The bottom level $j = L$ consists of $a^{L} = a^{\log_b n} = n^{\log_b a} = n^{\alpha}$ leaves, each of constant cost. Summing by levels:

$$
T(n) = \Theta(n^{\alpha}) + \sum_{j=0}^{L-1} a^{j} f\bigl(n/b^{j}\bigr)
$$

Everything now depends on how the level sums behave — a geometric-like series with ratio governed by $f$ versus $n^{\alpha}$.

**Case 1 ($f(n) = O(n^{\alpha - \epsilon})$: leaves dominate).** Substituting the bound,

$$
\sum_{j=0}^{L-1} a^{j}\,c\Bigl(\frac{n}{b^{j}}\Bigr)^{\alpha - \epsilon} = c\,n^{\alpha-\epsilon}\sum_{j=0}^{L-1}\Bigl(\frac{a}{b^{\alpha - \epsilon}}\Bigr)^{j} = c\,n^{\alpha-\epsilon}\sum_{j=0}^{L-1}\bigl(b^{\epsilon}\bigr)^{j}
$$

using $a = b^{\alpha}$. The geometric sum with ratio $b^{\epsilon} \gt 1$ is dominated by its last term, $\Theta\bigl(b^{\epsilon L}\bigr) = \Theta(n^{\epsilon})$, so the whole sum is $O(n^{\alpha - \epsilon}\cdot n^{\epsilon}) = O(n^{\alpha})$. Adding the leaf term $\Theta(n^{\alpha})$ gives $T(n) = \Theta(n^{\alpha})$.

**Case 2 ($f(n) = \Theta(n^{\alpha})$, i.e. $k = 0$: every level costs the same).** Then $a^{j}f(n/b^{j}) = \Theta\bigl(a^{j}(n/b^{j})^{\alpha}\bigr) = \Theta(n^{\alpha})$ for every $j$, so all $L = \log_b n$ levels contribute equally:

$$
T(n) = \Theta(n^{\alpha}) + \Theta\bigl(n^{\alpha}\log_b n\bigr) = \Theta\bigl(n^{\alpha}\log n\bigr)
$$

The general $k$ replaces the level sum by $\sum_j (\log(n/b^{j}))^{k} = \Theta(L^{k+1})$, giving $\Theta(n^{\alpha}\log^{k+1}n)$.

**Case 3 ($f(n) = \Omega(n^{\alpha+\epsilon})$ with regularity: the root dominates).** The regularity condition $af(n/b) \le cf(n)$ with $c \lt 1$ iterates to $a^{j}f(n/b^{j}) \le c^{j}f(n)$, so

$$
\sum_{j=0}^{L-1}a^{j}f(n/b^{j}) \le f(n)\sum_{j=0}^{\infty}c^{j} = \frac{f(n)}{1-c} = O\bigl(f(n)\bigr)
$$

and the $j = 0$ term alone gives the matching lower bound $\Omega(f(n))$. Since $f(n) = \Omega(n^{\alpha+\epsilon})$ dominates the leaf term $n^{\alpha}$, we get $T(n) = \Theta(f(n))$. $\blacksquare$

**Worked instances.**

| Recurrence | $a$, $b$, $\alpha = \log_b a$ | $f(n)$ vs $n^{\alpha}$ | Result |
|---|---|---|---|
| $T(n) = 2T(n/2) + n$ (merge sort) | $2, 2, \alpha = 1$ | $f = \Theta(n^{1})$: Case 2 | $\Theta(n\log n)$ |
| $T(n) = 4T(n/2) + n$ | $4, 2, \alpha = 2$ | $f = O(n^{2-\epsilon})$: Case 1 | $\Theta(n^{2})$ |
| $T(n) = 3T(n/4) + n^{2}$ | $3, 4, \alpha \approx 0.79$ | Case 3, regularity holds | $\Theta(n^{2})$ |
| $T(n) = 7T(n/2) + n^{2}$ (Strassen) | $7, 2, \alpha = \log_2 7$ | Case 1 | $\Theta(n^{\log_2 7}) \approx \Theta(n^{2.807})$ |
| $T(n) = 2T(n/2) + n\log n$ | $2, 2, \alpha = 1$ | Gap: $f/n^{\alpha} = \log n$ is not a power | Case 2 with $k=1$: $\Theta(n\log^{2}n)$ |

**Strategic reading.** The theorem is nothing but the recursion tree evaluated once and for all; the three cases are the three regimes of a geometric series (ratio $\gt 1$, $= 1$, $\lt 1$). Knowing the derivation means you can handle the recurrences the theorem misses — unequal splits, non-constant $a$, or an $f$ that violates regularity — by drawing the tree yourself.

### Proof 3.4 — Substitution, and the Strengthened-Hypothesis Trap

**Claim.** $T(n) = 2T(\lfloor n/2 \rfloor) + n$ with $T(1) = 1$ satisfies $T(n) = O(n\log n)$.

**Proof (substitution).** Guess $T(n) \le c\,n\log_2 n$ for $n \ge 2$ and a constant $c$ to be chosen. Assume it holds for all values below $n$ (strong induction). Then

$$
T(n) \le 2c\Bigl\lfloor \frac{n}{2}\Bigr\rfloor \log_2\Bigl\lfloor\frac{n}{2}\Bigr\rfloor + n \le 2c\,\frac{n}{2}\log_2\frac{n}{2} + n = cn(\log_2 n - 1) + n
$$

$$
= c\,n\log_2 n - cn + n \le c\,n\log_2 n \quad \text{whenever } c \ge 1
$$

The base case needs care since $\log_2 1 = 0$ makes the bound vacuous at $n=1$; start instead at $n = 2$ and $n = 3$, where $T(2) = 2T(1)+2 = 4 \le 2c$ and $T(3) = 2T(1) + 3 = 5 \le 3c\log_2 3 \approx 4.75c$, both satisfied by $c = 2$. Hence $T(n) = O(n\log n)$. $\blacksquare$ (A symmetric argument with $\lceil \cdot \rceil$ and a lower-bound guess gives $\Omega(n\log n)$, so in fact $T(n) = \Theta(n\log n)$.)

**The trap, part 1: a false guess that "verifies".** Suppose one guessed $T(n) = O(n)$ and tried $T(n) \le cn$:

$$
T(n) \le 2c\frac{n}{2} + n = cn + n
$$

A careless reader concludes "$cn + n = O(n)$, done" — but this is *not* $\le cn$, and the inductive step has failed. The hypothesis must be re-established in *exactly* the assumed form, with the same constant. Absorbing the surplus into the $O$-notation mid-induction is the single most common error in recurrence proofs, and here it would "prove" the false statement $T(n) = O(n)$.

**The trap, part 2: a true guess that needs strengthening.** For $T(n) = 2T(\lfloor n/2\rfloor) + 1$, the correct answer is $\Theta(n)$, yet the guess $T(n) \le cn$ gives

$$
T(n) \le 2c\frac{n}{2} + 1 = cn + 1 \not\le cn
$$

The step fails by a *constant*, not by a growing amount. The fix is to strengthen the hypothesis by subtracting a lower-order term: guess $T(n) \le cn - d$. Then

$$
T(n) \le 2\Bigl(c\frac{n}{2} - d\Bigr) + 1 = cn - 2d + 1 \le cn - d \quad \text{whenever } d \ge 1
$$

and the induction closes. Strengthening a statement to make its own induction work is counterintuitive but standard — the stronger hypothesis is also a stronger tool.

**Strategic reading.** Substitution is Topic 04's induction with an unknown constant carried along. Two disciplines make it reliable: (i) never let $O$-notation appear inside the inductive step — carry explicit constants; (ii) when the step fails by an additive constant, subtract a lower-order term from the guess rather than abandoning it.

### Proof 3.5 — Amortized Analysis of a Dynamic Array

**Setup.** A dynamic array supports `append`. When the backing store of capacity $s$ is full, a new store of capacity $2s$ is allocated and all $s$ elements are copied; otherwise the append costs $1$. A single append therefore costs $\Theta(n)$ in the worst case. We show $m$ appends cost $\Theta(m)$, i.e. $O(1)$ amortized per operation.

**Method 1: aggregate.** Starting from capacity $1$, resizes occur at sizes $1, 2, 4, \ldots, 2^{\lfloor \log_2 m\rfloor}$, copying that many elements each time. The total copy cost over $m$ appends is

$$
\sum_{j=0}^{\lfloor \log_2 m\rfloor} 2^{j} \lt 2^{\log_2 m + 1} = 2m
$$

by the geometric series. Adding the $m$ unit costs gives total $\lt 3m$, so the amortized cost per append is below $3 = O(1)$.

**Method 2: accounting.** Charge each append $3$ credits: $1$ pays for writing the new element, and $2$ are stored on it. When a resize copies the array, every element being copied has either just been charged or was appended after the previous resize; each such element carries $2$ unpaid credits, which pay for copying itself and for re-copying one older element. The invariant "every element added since the last resize holds $2$ credits" is maintained inductively, credits never go negative, so the charged cost $3m$ upper-bounds the actual cost.

**Method 3: potential.** Let $\Phi(D) = 2\cdot\mathrm{size}(D) - \mathrm{capacity}(D)$ immediately after an operation, which is $\ge 0$ whenever the array is at least half full — true for this scheme just after any resize. For a non-resizing append, the actual cost is $1$, size grows by $1$ and capacity is unchanged, so

$$
\hat{c} = 1 + \Delta\Phi = 1 + 2 = 3
$$

For a resizing append with size $s$ before (so capacity $s$, and capacity $2s$ after), the actual cost is $s + 1$ (copy $s$, write $1$), and

$$
\Delta\Phi = \bigl[2(s+1) - 2s\bigr] - \bigl[2s - s\bigr] = 2 - s \implies \hat{c} = (s+1) + (2 - s) = 3
$$

Every operation has amortized cost exactly $3$, and since $\Phi(D_0) = 0$ and $\Phi \ge 0$, Definition 2.9 gives total actual cost $\le 3m$. $\blacksquare$

**Why growth must be geometric.** If the array grew by a *constant* $k$ instead of doubling, resizes would occur every $k$ appends with cost $\Theta(n)$ each, totalling $\sum_{i} ik \approx m^{2}/(2k) = \Theta(m^{2})$ — quadratic. Geometric growth is exactly what makes the copy costs a convergent geometric series. This is why every practical vector, hash table, and replay buffer grows multiplicatively.

**Strategic reading.** All three methods prove the same theorem; the potential method is the most mechanical and generalizes best, because designing $\Phi$ is the entire creative act and the rest is arithmetic. Note that no probability appeared anywhere: amortized bounds are worst-case guarantees about sequences, not average-case claims.

### Proof 3.6 — Lower Bound: Comparison Sorting Is $\Omega(n\log n)$

**Model.** A comparison sort accesses the input only through queries "is $a_i \le a_j$?". Its execution on a fixed input length $n$ is described by a *decision tree*: internal nodes are comparisons, the two children correspond to the two answers, and each leaf outputs a permutation to apply.

**Step 1: the tree must have at least $n!$ leaves.** For the algorithm to be correct, every one of the $n!$ possible input orderings must be sorted correctly. Two different orderings require two different output permutations, and an execution reaching a given leaf always outputs the same permutation. Hence distinct orderings must reach distinct leaves, so the number of leaves $\ell$ satisfies $\ell \ge n!$ — a counting argument straight out of Topic 05.

**Step 2: a binary tree of height $h$ has at most $2^{h}$ leaves.** By induction on $h$: a tree of height $0$ has $1 = 2^{0}$ leaf; a tree of height $h+1$ has two subtrees of height $\le h$, so at most $2 \cdot 2^{h} = 2^{h+1}$ leaves.

**Step 3: combine.** The worst-case number of comparisons is the height $h$ of the tree, and

$$
2^{h} \ge \ell \ge n! \implies h \ge \log_2 (n!)
$$

By Proof 3.2, Claim C, $\log_2(n!) = \Theta(n\log n)$; explicitly, $\log_2(n!) \ge \frac{n}{2}\log_2\frac{n}{2} = \Omega(n\log n)$. Therefore every comparison sort needs $\Omega(n\log n)$ comparisons in the worst case. $\blacksquare$

**What the bound does and does not say.**

- It is *tight*: merge sort and heapsort achieve $\Theta(n\log n)$, so the model's limit is attained.
- It applies to the *model*, not to sorting as such. Counting sort, radix sort, and bucket sort read the keys' structure rather than merely comparing them, and run in $\Theta(n + k)$ or $\Theta(dn)$ — no contradiction, because they are outside the decision-tree model.
- The same technique bounds any problem with many distinguishable outputs: searching an ordered array needs $\Omega(\log n)$ comparisons (only $n+1$ outcomes, so the tree is shallower), and information-theoretic lower bounds in compression and learning have exactly this shape — $\log(\text{number of possible answers})$ bits must be extracted.

**Strategic reading.** Lower bounds require a *model of computation*: without one, "no algorithm can do better" is meaningless. The proof pattern is universal — count the required outputs, bound how much information one operation yields, divide.

## 4. Computational & Algorithmic Insights

**Where the time actually goes.** Asymptotics ranks algorithms as $n \to \infty$; at finite $n$ three effects routinely overturn the ranking.

| Effect | Consequence | Example |
|---|---|---|
| Hidden constants | A $\Theta(n\log n)$ routine can lose to $\Theta(n^{2})$ at small $n$ | Insertion sort beats merge sort below $n \approx 30$; production sorts switch over |
| Memory hierarchy | Cost model should count cache misses, not operations | Blocked matmul has identical FLOPs but several-fold better throughput |
| Parallelism | Wall-clock depends on *depth* (critical path), not total work | Prefix sum is $\Theta(n)$ work but $\Theta(\log n)$ depth |

The right response is not to abandon asymptotics but to state *which resource* is being counted: arithmetic operations, memory traffic (the I/O model), or parallel depth (the work–depth model). The FlashAttention line of work is precisely a re-analysis of attention in the I/O model: the FLOP count is unchanged, the bytes moved fall by an order of magnitude, and the wall-clock time follows the bytes.

**Cost of the core numerical kernels.**

| Operation | Time | Memory |
|---|---|---|
| Dot product, dimension $d$ | $\Theta(d)$ | $\Theta(1)$ |
| Matrix–vector, $n \times d$ | $\Theta(nd)$ | $\Theta(nd)$ |
| Matrix–matrix, $(m \times k)(k \times n)$ | $\Theta(mkn)$ classically | $\Theta(mk + kn + mn)$ |
| Strassen's algorithm, square $n$ | $\Theta(n^{\log_2 7}) \approx \Theta(n^{2.807})$ | Larger constants, less stable |
| Dense solve / LU factorization, $n \times n$ | $\Theta(n^{3})$ | $\Theta(n^{2})$ |
| Full eigen- or singular-value decomposition | $\Theta(n^{3})$ | $\Theta(n^{2})$ |
| FFT, length $n$ | $\Theta(n\log n)$ | $\Theta(n)$ |
| Sorting $n$ keys by comparison | $\Theta(n\log n)$, and no better | $\Theta(n)$ |

Two remarks. First, the exponent for matrix multiplication has a long theoretical history (the current record is near $2.37$), but algorithms below Strassen have constants so large that they are galactic — never used. Second, several apparently different problems are equivalent to matrix multiplication up to constants (inversion, LU, determinant), so its exponent governs a whole cluster of costs.

**Recurrences appearing in real algorithms.**

| Algorithm | Recurrence | Solution |
|---|---|---|
| Binary search | $T(n) = T(n/2) + \Theta(1)$ | $\Theta(\log n)$ |
| Merge sort, FFT | $T(n) = 2T(n/2) + \Theta(n)$ | $\Theta(n\log n)$ |
| Karatsuba multiplication | $T(n) = 3T(n/2) + \Theta(n)$ | $\Theta(n^{\log_2 3}) \approx \Theta(n^{1.585})$ |
| Strassen matmul | $T(n) = 7T(n/2) + \Theta(n^{2})$ | $\Theta(n^{2.807})$ |
| Quickselect (expected) | $T(n) = T(n/2) + \Theta(n)$ | $\Theta(n)$ |
| Quicksort (worst case) | $T(n) = T(n-1) + \Theta(n)$ | $\Theta(n^{2})$ |
| Unbalanced split (e.g. $1{:}9$) | $T(n) = T(n/10) + T(9n/10) + \Theta(n)$ | $\Theta(n\log n)$ — depth $\log_{10/9} n$ |

The last row carries the lesson that divide-and-conquer is robust: any *constant-fraction* split gives $\Theta(n\log n)$, because the tree depth changes only by a constant factor in the logarithm's base. Only splits that peel off $O(1)$ elements degrade to quadratic.

**Amortization in practice.** Dynamic arrays, hash tables with rehashing, union–find with path compression (amortized $O(\alpha(n))$, inverse-Ackermann — constant for all practical $n$), splay trees, and replay buffers all rely on amortized rather than per-operation bounds. When a system needs *predictable latency* rather than good throughput — real-time inference, for instance — amortized bounds are the wrong guarantee and incremental or de-amortized structures are used instead.

## 5. Real-World Physics & AI/ML Applications

**Physics and scientific computing.** The $N$-body problem is $\Theta(N^{2})$ by brute force; the Barnes–Hut tree code reduces it to $\Theta(N\log N)$ and the fast multipole method to $\Theta(N)$ — approximations whose error is controlled while the asymptotics improve, which is the standard bargain in scientific computing. Solving a sparse linear system by dense Gaussian elimination is $\Theta(n^{3})$, while conjugate gradients cost $\Theta(\sqrt{\kappa})$ iterations of $\Theta(\mathrm{nnz})$ work each — so the condition number $\kappa$ from Topic 03's numerical companion module enters the *complexity*, not just the accuracy. Explicit ODE solvers face a stability restriction $\Delta t = O(\Delta x^{2})$ for diffusion, making the total work scale as $\Theta(\Delta x^{-3})$ in one dimension; implicit schemes trade a linear solve per step for a much larger stable $\Delta t$.

**Attention: the quadratic wall.** For a sequence of $n$ tokens with model width $d$ and a single head:

$$
\underbrace{QK^{\top}}_{\Theta(n^{2}d)} \to \underbrace{\mathrm{softmax}}_{\Theta(n^{2})} \to \underbrace{(\cdot)V}_{\Theta(n^{2}d)} \implies \Theta(n^{2}d) \text{ time}, \ \Theta(n^{2}) \text{ score memory}
$$

while the position-wise feed-forward network costs $\Theta(nd^{2})$. Attention therefore dominates once $n \gtrsim d$: for $d = 4096$, a $2$K-token context is FFN-dominated while a $32$K-token context is attention-dominated. This single comparison explains the entire research programme around long context — sparse and sliding-window attention ($\Theta(nwd)$), linear/kernelized attention ($\Theta(nd^{2})$), state-space models ($\Theta(nd\log n)$ or $\Theta(nd)$), and IO-aware exact attention (FlashAttention: same $\Theta(n^{2}d)$ FLOPs, but $\Theta(n^{2})$ score memory reduced to $\Theta(n)$ by never materializing the matrix).

**AI/ML applications.**

- **Training cost is linear in parameters and tokens.** A forward pass costs about $2N$ FLOPs per token for $N$ parameters (one multiply and one add per parameter), and the backward pass about twice that, giving the standard $C \approx 6ND$ estimate for training on $D$ tokens. Inference decoding costs $\approx 2N$ per generated token plus attention over the growing KV cache. These are pure product-rule counts (Topic 05) wrapped in asymptotic language, and they are what compute-budget and scaling-law analyses actually use.
- **Backpropagation is a constant factor, not a factor of $N$.** The Baur–Strassen theorem states that the gradient of a function computed by an arithmetic circuit of size $s$ can be computed by a circuit of size $O(s)$ — reverse-mode autodiff computes *all* partial derivatives in a constant multiple (typically $2$–$3\times$) of the forward cost. Numerical differentiation would cost $\Theta(N)$ forward passes; if that were the only option, training a $10^{9}$-parameter model would be impossible.
- **Memory, not time, is often the binding constraint.** Activations for an $L$-layer network cost $\Theta(L \cdot b \cdot n \cdot d)$ to store for the backward pass. Gradient checkpointing recomputes instead of storing: keeping $\sqrt{L}$ checkpoints reduces activation memory to $\Theta(\sqrt{L})$ segments at the cost of one extra forward pass — a textbook time–memory trade-off with an optimum at $\sqrt{L}$ found by minimizing $\Theta(L/k + k)$ over $k$. Optimizer state adds its own constant: SGD needs $N$ words, momentum $2N$, Adam $3N$ (parameters plus two moments).
- **Nearest neighbours and the curse of dimensionality.** Exact $k$-NN over $m$ points in $d$ dimensions is $\Theta(md)$ per query by brute force; KD-trees achieve $O(\log m)$ only while $d$ is small, degrading to $\Theta(m)$ once $d \gtrsim \log m$. This is why production retrieval uses approximate methods — LSH, HNSW graphs, product quantization — trading exactness for sub-linear query time.
- **Convolution versus attention versus MLP.** A convolution with kernel size $k$ over $n$ positions and $c$ channels costs $\Theta(nkc^{2})$ — linear in sequence length, which is exactly the property attention gives up in exchange for a global receptive field in one layer. Reading the three costs $\Theta(nkc^{2})$, $\Theta(n^{2}d)$, $\Theta(nd^{2})$ side by side is the cleanest way to understand architectural trade-offs.
- **Batch size, throughput, and the roofline.** Doubling batch size doubles FLOPs but often less than doubles time, because small batches are memory-bandwidth-bound rather than compute-bound. The arithmetic intensity (FLOPs per byte moved) determines which side of the roofline a kernel sits on, and asymptotic analysis in the I/O model — not the operation-count model — predicts the wall clock.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source |
|---|---|
| Definitions of $O$, $\Omega$, $\Theta$, $o$, $\omega$ with witness constants | CLRS, *Introduction to Algorithms*, Ch. 3; Rosen, *Discrete Mathematics*, Ch. 3.2 |
| Careful history and pitfalls of $O$-notation | Knuth, *TAOCP* Vol. 1, Sec. 1.2.11; Knuth (1976), "Big Omicron and Big Omega and Big Theta" |
| Growth hierarchy, Stirling's approximation | Graham, Knuth & Patashnik, *Concrete Mathematics*, Ch. 9 |
| Substitution, recursion trees, the Master theorem | CLRS Ch. 4; Kleinberg & Tardos, Ch. 5 |
| Akra–Bazzi for recurrences the Master theorem misses | Leighton (1996), MIT technical note; CLRS Ch. 4 problems |
| Amortized analysis: aggregate, accounting, potential | CLRS Ch. 16; Tarjan (1985), "Amortized computational complexity" |
| Decision-tree lower bound for sorting | CLRS Ch. 8.1; Knuth, *TAOCP* Vol. 3, Sec. 5.3.1 |
| Matrix multiplication exponents and stability | Golub & Van Loan, *Matrix Computations*, Ch. 1; Strassen (1969) |
| Reverse-mode autodiff cost (gradient is $O$(function)) | Baur & Strassen (1983); Griewank & Walther, *Evaluating Derivatives* |
| Gradient checkpointing and the $\sqrt{L}$ trade-off | Chen et al. (2016), "Training Deep Nets with Sublinear Memory Cost" |
| I/O-aware analysis of attention | Dao et al. (2022), *FlashAttention* |
| Transformer FLOP accounting and scaling laws | Kaplan et al. (2020); Hoffmann et al. (2022), *Chinchilla* |

**Reading path.** CLRS Ch. 3–4 for the definitions and recurrence machinery, then Ch. 16 for amortization and Ch. 8.1 for the sorting lower bound — that trio covers the classical core. *Concrete Mathematics* Ch. 9 is the reference when asymptotic *expansions* (not just orders) are needed. For the machine learning side, read the FlashAttention paper as a case study in choosing the right cost model, and the Chinchilla paper as a case study in what the $6ND$ estimate buys you. Topic 04 supplied the induction underlying substitution, and Topic 05 supplied the counting underlying the $\log(n!)$ lower bound; this module closes the mathematical-reasoning sequence by turning both into statements about running programs.